In [ ]:
import pandas as pd
import re

jd = pd.read_csv("../data/raw/jd_full_text_20260810.csv")
meta = pd.read_csv("../data/processed/jobs_cleaned_20260808.csv")
df = jd.merge(meta, on="id", how="left")
df["jd_lower"] = df["jd_text"].str.lower()
print(df.shape)

In [ ]:
NEGATIVE = [
    r"\bno\s+(?:visa\s+)?sponsorship\b",
    r"\b(?:not|cannot|can'?t|do(?:es)?\s+not|don'?t|won'?t|isn'?t|aren'?t)\s+(?:be\s+)?(?:able\s+to\s+)?(?:offer|provide|support|consider)\s+(?:visa\s+)?sponsorship\b",
    r"\bsponsorship\s+(?:is\s+)?(?:not|isn'?t)\s+(?:available|offered|provided)\b",
    r"\b(?:visa\s+)?sponsorship\s+isn'?t\s+available\b",
    r"\bwithout\s+(?:visa\s+)?sponsorship\b",
    r"\b(?:not|unable\s+to|cannot|can'?t)\s+(?:be\s+able\s+to\s+)?sponsor\b",
    r"\bunable\s+to\s+sponsor\b",
    r"\bdo(?:es)?\s+not\s+sponsor\b",
    r"\bwill\s+not\s+sponsor\b",
    r"\bonly\s+accept\s+applicants?\s+who\s+have\s+(?:a\s+)?right\s+to\s+work\b",
    r"\bmust\s+(?:already\s+)?have\s+(?:the\s+|a\s+)?right\s+to\s+work\b",
    r"\bmust\s+be\s+eligible\s+to\s+work\b",
    r"\b(?:existing|current)\s+right\s+to\s+work\b",
    r"\bunable\s+to\s+(?:offer|provide|support)\s+(?:visa\s+)?sponsorship\b",
]

POSITIVE = [
    r"\b(?:are\s+)?able\s+to\s+offer\s+(?:visa\s+)?sponsorship\b",
    r"\bsponsorship\s+(?:is\s+)?available\b",
    r"\bvisa\s+sponsorship\s+available\b",
    r"\bvisa\s+sponsorship\s+(?:and|where|,)",
    r"\bincluding\s+visa\s+sponsorship\b",
    r"\b(?:licen[cs]ed|registered)\s+sponsor\b",
    r"\bskilled\s+worker\s+visa\b",
    r"\btier\s*2\b",
    r"\b(?:can|will|happy\s+to)\s+sponsor\s+(?:you|candidates?|applicants?|visa)\b",
    r"\bsponsorship\s+(?:will\s+be\s+)?considered\b",
    r"\bwe\s+(?:can\s+)?offer\s+(?:visa\s+)?sponsorship\b",
]

TRIGGER = r"\b(sponsorship|sponsor|visa|right to work|immigration)\b"

def classify(text):
    if any(re.search(p, text) for p in NEGATIVE):
        return "no_sponsorship"
    if any(re.search(p, text) for p in POSITIVE):
        return "sponsorship_offered"
    if re.search(TRIGGER, text):
        return "mentioned_unclear"
    return "not_mentioned"

df["sponsor_status"] = df["jd_lower"].apply(classify)
print(df["sponsor_status"].value_counts())

In [ ]:
def show_sponsor_context(status, n=10, width=120):
    sub = df[df["sponsor_status"] == status]
    print(f"=== {status} ({len(sub)} 条) ===")
    for t in sub["jd_lower"].head(n):
        m = re.search(TRIGGER, t)
        if m:
            s = max(0, m.start() - width)
            e = min(len(t), m.end() + width)
            print("…" + t[s:e].replace("\n", " ") + "…\n")

show_sponsor_context("no_sponsorship")

In [ ]:
show_sponsor_context("sponsorship_offered")

In [ ]:
show_sponsor_context("mentioned_unclear")

In [ ]:
show_sponsor_context("sponsorship_offered")

In [ ]:
show_sponsor_context("no_sponsorship")

In [ ]:
show_sponsor_context("mentioned_unclear")

In [ ]:
jd214 = pd.read_csv("../data/raw/jd_full_text_20260811.csv")
df = jd214.merge(meta, on="id", how="left")
df["jd_lower"] = df["jd_text"].str.lower()
df["sponsor_status"] = df["jd_lower"].apply(classify)

print(df.shape)
print(df["sponsor_status"].value_counts())
print()
print((df["sponsor_status"].value_counts(normalize=True) * 100).round(1))

In [ ]:
SENIORITY = [
    ("graduate", r"\b(?:graduate|intern|internship|placement|trainee|apprentice)\b"),
    ("junior",   r"\b(?:junior|entry.level|jr\.?)\b"),
    ("senior",   r"\b(?:senior|snr\.?|sr\.?)\b"),
    ("lead",     r"\b(?:lead|principal|staff|head\s+of|director|chief)\b"),
]

def get_seniority(t):
    for label, pat in SENIORITY:
        if re.search(pat, t):
            return label
    return "unspecified"

df["seniority"] = df["title"].str.lower().apply(get_seniority)
print(df["seniority"].value_counts())
print()
print(pd.crosstab(df["seniority"], df["sponsor_status"]))

In [ ]:
cols = ["id", "title", "company_name", "region", "seniority",
        "sponsor_status", "salary_min", "salary_max", "salary_is_predicted"]
df[cols].to_csv("../data/processed/sponsorship_20260811.csv", index=False)
print(df[cols].shape)